# Fine-Tuning BERT for Question Answering on SQuAD v1.1
**Name:** YourName  |  **Course/Assignment:** Fine-Tuning Transformer Models for Question Answering

> **Before running:** go to `Runtime -> Change runtime type -> T4 GPU`. Then run all cells top to bottom.
> To share: `Share -> General access -> Anyone with the link -> Viewer`.

## 1. Introduction: Classification vs. Question Answering

**Classification** (e.g. sentiment analysis) takes an input text and predicts **one label from a fixed set** (positive / negative, spam / not spam). The output space is small and decided in advance. The model has one output head that produces one probability distribution over the classes.

**Extractive Question Answering** takes a **question + a context passage** and must find the **span of text inside the context** that answers the question. There is no fixed label set; the answer is different for every example. The model does not generate text. Instead, for every token in the context it predicts two scores:

* the probability that this token is the **start** of the answer,
* the probability that this token is the **end** of the answer.

So QA is really **token-level classification**: "which token is the start?" and "which token is the end?" This is why we need to convert the answer's *character* position into *token* positions during preprocessing, and why the evaluation uses **Exact Match** and **F1** (word overlap) rather than plain accuracy.

| | Classification | Extractive QA |
|---|---|---|
| Input | one text (or a pair) | question + context |
| Output | one label | start & end token positions (a span) |
| Output space | fixed, small | depends on the context length |
| Metric | accuracy / F1 over labels | Exact Match, token-level F1 |

## 2. Implementation

### Step 1 - Environment setup
We install:
* `transformers`: the BERT model, tokenizer, and `Trainer`
* `datasets`: to download SQuAD
* `evaluate`: gives us the official SQuAD metric (EM and F1)
* `accelerate`: required by `Trainer` for GPU training

In [ ]:
!pip install -q transformers datasets evaluate accelerate

In [ ]:
import torch, numpy as np, collections
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

**Quick-run switch.** Training on all 88k examples for 2 epochs takes roughly 1.5-2 hours on a free T4 GPU.
If you want a fast test first, set `USE_SUBSET = True`. For your **final submission set it to `False`** to get the
standard ~80+ EM / ~88 F1 results.

In [ ]:
USE_SUBSET = False      # True = fast test run (small data), False = full training
N_TRAIN = 5000          # used only when USE_SUBSET = True
N_VAL = 1000

### Step 2 - Load and explore the dataset
SQuAD v1.1 contains Wikipedia paragraphs, with questions written by crowd-workers. Every answer is a span of the paragraph.
It has ~87.6k training and ~10.6k validation examples.

In [ ]:
from datasets import load_dataset
dataset = load_dataset("rajpurkar/squad")
print(dataset)

In [ ]:
# Look at one training example
example = dataset["train"][0]
for k, v in example.items():
    print(f"{k}: {v}\n")

Each example has:
* `question`: the question text
* `context`: the paragraph containing the answer
* `answers`: a dict with `text` (the answer string) and `answer_start` (the **character** index in the context where the answer begins)

Let's verify that the character position really points to the answer:

In [ ]:
ctx = example["context"]
start = example["answers"]["answer_start"][0]
text = example["answers"]["text"][0]
print("Answer from context slice:", ctx[start:start+len(text)])
print("Answer text field        :", text)

In [ ]:
# Basic statistics
import pandas as pd
df = pd.DataFrame(dataset["train"])
print("Train size:", len(dataset["train"]), "| Validation size:", len(dataset["validation"]))
print("Avg context length (words):", df["context"].str.split().str.len().mean().round(1))
print("Avg question length (words):", df["question"].str.split().str.len().mean().round(1))

In [ ]:
if USE_SUBSET:
    dataset["train"] = dataset["train"].shuffle(seed=42).select(range(N_TRAIN))
    dataset["validation"] = dataset["validation"].shuffle(seed=42).select(range(N_VAL))
print({k: len(v) for k, v in dataset.items()})

### Step 3 - Load the tokenizer and model
`AutoModelForQuestionAnswering` loads `bert-base-uncased` and adds a **new, randomly initialised QA head**: a single linear layer
that maps every token's hidden vector (768 dims) to 2 numbers: the *start logit* and the *end logit*.
You will see a warning that some weights are newly initialised. That is expected because they are what we are about to train.

In [ ]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

### Step 4 - Tokenization and preprocessing (the most important step)

BERT takes the input as `[CLS] question [SEP] context [SEP]` and has a **maximum length of 512 tokens** (we use 384 to save memory).
Two problems must be solved:

**Problem 1 - Long contexts.** Some contexts are longer than 384 tokens. If we just cut them, the answer might be lost.
Solution: split the context into **overlapping windows** (`stride=128`, `return_overflowing_tokens=True`). One example can therefore produce several training features.

**Problem 2 - Labels are in characters, the model needs tokens.** SQuAD gives `answer_start` as a character index, but the model predicts token indexes.
Solution: use `return_offsets_mapping=True`, which tells us, for each token, the `(start_char, end_char)` it covers in the original text. We then find
which tokens cover the answer's start and end characters.

If the answer is **not inside a given window**, we label that window with `(0, 0)`, which points to the `[CLS]` token, meaning "no answer here".

In [ ]:
max_length = 384   # max tokens per feature
doc_stride = 128   # overlap between windows of a long context

def prepare_train_features(examples):
    # Remove leading whitespace in questions (it can confuse the tokenizer)
    questions = [q.lstrip() for q in examples["question"]]

    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",        # only cut the context, never the question
        stride=doc_stride,
        return_overflowing_tokens=True,  # long context -> several features
        return_offsets_mapping=True,     # token -> character positions
        padding="max_length",
    )

    # Which original example each feature came from
    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions, end_positions = [], []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        # sequence_ids: 0 = question tokens, 1 = context tokens, None = special tokens
        sequence_ids = tokenized.sequence_ids(i)

        answers = examples["answers"][sample_map[i]]
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        # Find where the context begins and ends inside this feature
        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1
        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        # Is the answer fully inside this window?
        if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
            start_positions.append(cls_index)   # answer not here -> point to [CLS]
            end_positions.append(cls_index)
        else:
            # move start pointer to the token containing the first answer character
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                token_start_index += 1
            start_positions.append(token_start_index - 1)
            # move end pointer to the token containing the last answer character
            while offsets[token_end_index][1] >= end_char:
                token_end_index -= 1
            end_positions.append(token_end_index + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

In [ ]:
train_dataset = dataset["train"].map(
    prepare_train_features,
    batched=True,
    remove_columns=dataset["train"].column_names,
)
print(train_dataset)

**Sanity check:** decode the tokens between the labelled start and end positions. They should equal the real answer.

In [ ]:
f = train_dataset[0]
decoded = tokenizer.decode(f["input_ids"][f["start_positions"]: f["end_positions"] + 1])
print("Decoded answer from labels:", decoded)
print("True answer               :", dataset["train"][0]["answers"]["text"][0])

**Validation preprocessing.** For validation we do not need labels for training, but we need to *map predictions back to the original text*.
So we keep the `offset_mapping` (set to `None` for non-context tokens so we never predict from the question) and an `example_id` for each feature.

In [ ]:
def prepare_validation_features(examples):
    questions = [q.lstrip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_map = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []

    for i in range(len(tokenized["input_ids"])):
        sequence_ids = tokenized.sequence_ids(i)
        tokenized["example_id"].append(examples["id"][sample_map[i]])
        # keep offsets only for context tokens
        tokenized["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]
    return tokenized

validation_dataset = dataset["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=dataset["validation"].column_names,
)
print(validation_dataset)

### Step 5 - Fine-tuning with the Hugging Face `Trainer`
Settings (as recommended in the assignment):
* **Epochs:** 2
* **Batch size:** 16
* **Learning rate:** 3e-5
* `weight_decay=0.01`: light regularisation
* `fp16=True`: mixed precision, which makes training faster and uses less GPU memory

During training the model minimises the average cross-entropy of the start-position prediction and the end-position prediction.

In [ ]:
from transformers import TrainingArguments, Trainer, default_data_collator

args = TrainingArguments(
    output_dir="bert-finetuned-squad",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=200,
    save_strategy="no",       # do not fill Colab disk with checkpoints
    report_to="none",         # disable wandb prompts
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=default_data_collator,   # data is already padded to max_length
    processing_class=tokenizer,
)

In [ ]:
train_result = trainer.train()
print(train_result.metrics)

### Step 6 - Evaluation
Evaluation has three parts:

1. **Predict** start/end logits for every validation feature.
2. **Post-process:** for each original question, look at the best start/end token pairs across all its windows, discard invalid ones
   (end before start, answer too long, positions in the question), pick the highest-scoring valid pair, and convert it back to text with `offset_mapping`.
3. **Score** with the official SQuAD metric:
   * **Exact Match (EM):** % of predictions that match a ground-truth answer *exactly* (after normalisation).
   * **F1:** word-level overlap between prediction and ground truth (gives partial credit).

In [ ]:
# 1) Predict logits
raw_predictions = trainer.predict(validation_dataset.remove_columns(["example_id", "offset_mapping"]))
start_logits, end_logits = raw_predictions.predictions
print(start_logits.shape, end_logits.shape)

In [ ]:
# 2) Post-processing: logits -> answer text
n_best = 20            # consider the top-20 start and end tokens
max_answer_length = 30 # answers longer than 30 tokens are discarded

def postprocess_predictions(examples, features, start_logits, end_logits):
    # group features by the example they belong to
    features_per_example = collections.defaultdict(list)
    for i, feat in enumerate(features):
        features_per_example[feat["example_id"]].append(i)

    predicted = {}
    for example in examples:
        context = example["context"]
        best = {"score": -1e9, "text": ""}

        for fi in features_per_example[example["id"]]:
            s_logits = start_logits[fi]
            e_logits = end_logits[fi]
            offsets = features[fi]["offset_mapping"]

            start_idxs = np.argsort(s_logits)[-1: -n_best - 1: -1].tolist()
            end_idxs = np.argsort(e_logits)[-1: -n_best - 1: -1].tolist()

            for s in start_idxs:
                for e in end_idxs:
                    # skip positions outside the context
                    if offsets[s] is None or offsets[e] is None:
                        continue
                    # skip invalid spans
                    if e < s or e - s + 1 > max_answer_length:
                        continue
                    score = s_logits[s] + e_logits[e]
                    if score > best["score"]:
                        best = {"score": score,
                                "text": context[offsets[s][0]: offsets[e][1]]}

        predicted[example["id"]] = best["text"]
    return predicted

predicted_answers = postprocess_predictions(
    dataset["validation"], validation_dataset, start_logits, end_logits
)
print("Predictions made:", len(predicted_answers))

In [ ]:
# 3) Score with the official SQuAD metric
import evaluate
squad_metric = evaluate.load("squad")

predictions = [{"id": k, "prediction_text": v} for k, v in predicted_answers.items()]
references = [{"id": ex["id"], "answers": ex["answers"]} for ex in dataset["validation"]]

results = squad_metric.compute(predictions=predictions, references=references)
print("Validation Exact Match :", round(results["exact_match"], 2))
print("Validation F1 Score    :", round(results["f1"], 2))

## 3. Results

The cell below prints a clean results table. For reference, a well-trained `bert-base-uncased` on the full SQuAD v1.1
usually reaches about **EM = 80-81** and **F1 = 88**.

In [ ]:
results_df = pd.DataFrame({
    "Metric": ["Exact Match (EM)", "F1 Score"],
    "Validation Score": [round(results["exact_match"], 2), round(results["f1"], 2)],
})
results_df

### Custom question-answer tests
We test the fine-tuned model on our own questions. Manual method: tokenize, run the model, take the `argmax` of the start and end logits,
and decode the tokens between them. (This is exactly the idea in the assignment's example; the extra step is converting logits into an answer.)

In [ ]:
def answer_question(question, context):
    model.eval()
    inputs = tokenizer(question, context, return_tensors="pt", truncation="only_second", max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)

    start_idx = int(torch.argmax(outputs.start_logits))
    end_idx = int(torch.argmax(outputs.end_logits))
    if end_idx < start_idx:
        return "(no valid answer found)"
    return tokenizer.decode(inputs["input_ids"][0][start_idx:end_idx + 1], skip_special_tokens=True)

tests = [
    ("Who developed the theory of relativity?",
     "Albert Einstein developed the theory of relativity in the early 20th century."),
    ("Where is the Eiffel Tower located?",
     "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It was completed in 1889."),
    ("When was the Eiffel Tower completed?",
     "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It was completed in 1889."),
    ("What is the capital of Bangladesh?",
     "Dhaka is the capital and largest city of Bangladesh. It is located on the banks of the Buriganga River."),
    ("Which river is Dhaka located on?",
     "Dhaka is the capital and largest city of Bangladesh. It is located on the banks of the Buriganga River."),
]

for q, c in tests:
    print("Q:", q)
    print("A:", answer_question(q, c))
    print("-" * 60)

The same test using the high-level Hugging Face `pipeline`, which does tokenization, long-context handling, and post-processing automatically
and also returns a confidence score:

In [ ]:
from transformers import pipeline

qa = pipeline("question-answering", model=trainer.model, tokenizer=tokenizer,
              device=0 if torch.cuda.is_available() else -1)

out = qa(question="Who developed the theory of relativity?",
         context="Albert Einstein developed the theory of relativity in the early 20th century.")
print(out)

In [ ]:
# Optional: save the model so you can reload it later
trainer.save_model("bert-finetuned-squad-final")
tokenizer.save_pretrained("bert-finetuned-squad-final")

## 4. Reflection

*(Edit this so it reflects your own experience and your actual numbers.)*

In this assignment I learned that extractive question answering is not a normal classification task: instead of one label, the model predicts a start and an end token inside the context. The hardest part was preprocessing, because SQuAD gives answer positions in characters while BERT needs token positions, so I had to use offset mappings to convert between them. Handling long contexts with a sliding window (`stride`) was another challenge, since an answer may fall in only one of several overlapping windows, and the other windows must be labelled with the `[CLS]` position. Post-processing the logits back into text, by picking the best valid start/end pair and rejecting spans that are too long or outside the context, taught me that model evaluation is more than just calling `predict`. After fine-tuning, my model reached about **EM = XX.X** and **F1 = XX.X** on the validation set (replace with your numbers), and it answered my custom questions correctly, which showed how effectively a pre-trained model can be adapted with only two epochs of training. Overall, I gained practical experience with the Hugging Face `datasets`, `Trainer`, and `evaluate` libraries and a better understanding of how transformers handle span-extraction tasks.